https://github.com/facebookresearch/faiss/wiki/Running-on-GPUs

# pyserini love java (install java)
!sudo apt-get update
!sudo apt-get install openjdk-21-jre -y
!java -version
!sudo  update-java-alternatives --list
!export JAVA_HOME='/usr/lib/jvm/java-1.21.0-openjdk-amd64'
import os
# linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-1.21.0-openjdk-amd64"
# installing pyserini
#https://github.com/castorini/pyserini
!python --version
!pip install faiss-cpu -q
!pip install pyserini -q
!pip install transformers==4.33.3 -q
# install numpy==1.26.4 to fix package conflit 
import numpy 
numpy.__version__
#!pip show numpy
!pip install --upgrade numpy==1.26.4

# 100 bm25 mr.Tydi
!python -m pyserini.search.lucene \
  --threads 16 --batch-size 128 \
  --language ar \
  --topics mrtydi-v1.1-arabic-test \
  --index mrtydi-v1.1-ar \
  --output run.mrtydi.bm25.ar.test.txt --bm25 --hits 1000 
!python -m pyserini.eval.trec_eval \
  -c -m recip_rank.100 -m recall.10,100 mrtydi-v1.1-arabic-test \
  run.mrtydi.bm25.ar.test.txt

In [ ]:
!pip install sentence-transformers -q
!pip install huggingface_hub -q

In [1]:
from huggingface_hub import hf_hub_download
import torch
organization_dataset_id= "hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning"
collection_name= "mrtydi-collection-arabic-preprocess-for-araelectra"
queries_name= "mrtydi-queries-arabic-preprocess-for-araelectra"
qerl_file ="qrels.test.mrtydi.txt"
bm25_100condidatxquery ="run.mrtydi.bm25.ar.test.txt"
#zero-shot evaluation first stage mr.Tydi
hf_hub_download(repo_id=organization_dataset_id, filename=qerl_file, local_dir="./", repo_type="dataset")
hf_hub_download(repo_id=organization_dataset_id, filename=bm25_100condidatxquery, local_dir="./", repo_type="dataset")
ranked_list_mrTydi=["FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-KDaradpr.tsv", "FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-NOKDaradpr.tsv", "FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-mmarco-Arabic-AraElectra-bi-encoder-KD-v1.tsv", "FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-mmarco-Arabic-AraElectra-bi-encoder-NoKD-v1.tsv", "FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-mmarco-Arabic-mMiniLML-bi-encoder-KD-v1.tsv", "FirstStage_mrTydi/FirstStagemrTydi-collection-arabic-embedding-mmarco-Arabic-mMiniLML-bi-encoder-NoKD-v1.tsv", ]

from datasets import load_dataset
datasetC = load_dataset(organization_dataset_id, collection_name, trust_remote_code=True)
print(datasetC['train'][100])
print(datasetC)

from datasets import load_dataset
datasetQ = load_dataset(organization_dataset_id, queries_name, trust_remote_code=True)
datasetQ = datasetQ["train"]
print(datasetQ)
print(datasetQ[0])

qrels.test.mrtydi.txt: 0.00B [00:00, ?B/s]

run.mrtydi.bm25.ar.test.txt: 0.00B [00:00, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

mrtydi-collection-arabic-preprocess-for-(…):   0%|          | 0.00/215M [00:00<?, ?B/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hatemestinbejaia/ExperimentDATA_knowledge_distillation_vs_fine_tuning' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


{'docid': '38#2', 'title': 'رياضيات', 'text': 'و لقد نشأ علم الرياضيات عندما قاس الإنسان ما شاهده من ظواهر طبيعية و بناء على فطرة وخاصية فيه ألا و هي اهتمامه بقياس كل ما حوله إلى جانب احتياجاته العملية فهكذا كانت هناك ضرورة لقياس قسمة الأقوات ( الطعام ) بين أفراد العائلة و قياس الوقت و الفصول و المحاصيل الزراعية و تقسيم الأراضي و غنائم الحملات الحربية و المحاسبة للتمكن من الإتجار إلى جانب علم الملاحة حيث الاهتداء بالنجوم في السفر و الترحال للتجارة و السياحة والقياسات اللازمة لتشييد الأبنية و المدن .'}
DatasetDict({
    train: Dataset({
        features: ['docid', 'title', 'text'],
        num_rows: 2106586
    })
})


mrtydi-queries-arabic-preprocess-for-ara(…):   0%|          | 0.00/41.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1081 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'text'],
    num_rows: 1081
})
{'id': 15513, 'text': 'كم عدد مرات فوز الأوروغواي ببطولة كاس العالم لكرو القدم ؟'}


In [6]:
import numpy as np
# compute lengths
lengths = [len(text.split()) for text in datasetC['train']["text"]]

min_len = np.min(lengths)
max_len = np.max(lengths)
mean_len = np.mean(lengths)
std = np.std(lengths)
print(std)
print("Min text length:", min_len)
print("Max text length:", max_len)
print("Mean text length:", mean_len)

73.76273187284924
Min text length: 0
Max text length: 9317
Mean text length: 59.49711001592149


In [7]:
# compute lengths
lengths = [len(text.split()) for text in datasetQ["text"]]

min_len = np.min(lengths)
max_len = np.max(lengths)
mean_len = np.mean(lengths)
std = np.std(lengths)
print(std)
print("Min text length:", min_len)
print("Max text length:", max_len)
print("Mean text length:", mean_len)

2.128397893054066
Min text length: 3
Max text length: 18
Mean text length: 6.62627197039778


In [ ]:
for i in range(len(ranked_list_mrTydi)):
    hf_hub_download(repo_id=organization_dataset_id, filename=ranked_list_mrTydi[i], local_dir="./", repo_type="dataset")

for i in ranked_list_mrTydi:
        print("*****************************************")
        print(i)
        print("*****************************************")
        !python -m pyserini.eval.trec_eval   -m  recip_rank.100 -m recall.100 \
            qrels.test.mrtydi.txt \
            {i}
        #break

In [ ]:
!head run.mrtydi.bm25.ar.test.txt
import pandas as pd

# Load the run file (space-separated)
df = pd.read_csv(
    "run.mrtydi.bm25.ar.test.txt",
    sep=r"\s+",
    header=None,
    names=["qid", "Q0", "docid", "rank", "score", "system"]
)

# Filter the target row
df0 = df[["qid", "docid", "rank"]] 
# Save 
df0.to_csv("mrtydi_bm25_ar_test.tsv", sep="\t", index=False, header=False)
df0.head()

In [ ]:
import os
files = ["./FirstStage_mrTydi/"+f for f in os.listdir("./FirstStage_mrTydi/")]
files.append('mrtydi_bm25_ar_test.tsv')
print(files)
print(len(files))

In [ ]:
import pandas as pd
from collections import defaultdict
import glob

def load_ranking_file(path):
    """Load a ranking file of format: qid, docid, rank."""
    return pd.read_csv(path, sep="\t", names=["qid", "docid", "rank"])

def rrf(ranking_lists, k=60):
    """Apply RRF fusion to a list of ranking lists (docid lists)."""
    scores = defaultdict(float)
    for ranking in ranking_lists:
        for rank, docid in enumerate(ranking):
            scores[docid] += 1.0 / (k + rank + 1)
    # Sort documents by RRF score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rrf_fusion(run_files, output_path=None, k=60, top_k=500):
    """
    Apply RRF to multiple ranking files with identical structure.
    
    run_files: list of file paths
    output_path: where to save fused run (optional)
    Returns: fused DataFrame
    """
    
    # 1) Load all runs
    runs = [load_ranking_file(f) for f in run_files]

    # All topics from first file
    topics = runs[0]["qid"].unique()

    fused_rows = []

    for qid in topics:
        ranking_lists = []

        # 2) Extract docids per topic from each run
        for df in runs:
            docs = df[df["qid"] == qid].sort_values("rank")["docid"].tolist()
            ranking_lists.append(docs)

        # 3) Apply RRF
        fused_docs = rrf(ranking_lists, k=k)

        # 4) Keep top_k
        for rank, (docid, score) in enumerate(fused_docs[:top_k], start=1):
            fused_rows.append([qid, docid, rank])

    # Create DataFrame
    fused_df = pd.DataFrame(fused_rows, columns=["qid", "docid", "rank"])

    # Optional save
    if output_path:
        fused_df.to_csv(output_path, sep="\t", index=False, header=False)
        print(f"Saved RRF run to: {output_path}")

    return fused_df
rrf_fusion(files, output_path="mrTydi_bm25_6faiss_rrf.txt")

In [ ]:
import tqdm
list_of_models = [ "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
              "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraElectra-cross-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-NoKD-v1", 
              "hatemestinbejaia/mmarco-Arabic-mMiniLML-cross-encoder-KD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-NoKD-v1",
              "hatemestinbejaia/mmarco-Arabic-AraDPR-cross-encoder-KD-v1",
            ]

In [ ]:
# Extract columns as lists
docids = datasetC['train']['docid']
texts = datasetC['train']['text']

# Build dictionary in one line
corpus = dict(zip(docids, texts))

In [ ]:
queries = {}
for i in tqdm.trange(len(datasetQ)):
    queries[datasetQ['id'][i]]= datasetQ['text'][i]
    #break

In [ ]:
import pandas as pd
#defaut number of condidat document K is 1000
def load_run(path, k=1000):
    print('Loading run...')
    run = pd.read_csv(path, delim_whitespace=True, header=None) # remove 
    #print(run.head(10))
    run = run[run[2] <= k] 
    #print(run.head(10))
    run = run.groupby(0)[1].apply(list).to_dict()
    return run

In [ ]:
import operator
import torch
def reranking_crossEncoder(query_id, passage_list_id, threshold):
    corpora=[]
    query = queries[query_id]
    X=[]
    for j in passage_list_id : 
        corpora.append([query, corpus[j]])
        X.append(j)
    with torch.no_grad():scores = model.predict(corpora, batch_size=8, show_progress_bar=False)
    #print(scores)
    x = dict(zip(X, scores))
    #print(x)
    xr = dict(sorted(x.items(), key=operator.itemgetter(1), reverse=True))
    # Filter dictionary to keep only scores > threshold
    #xr_filtered = {doc: score for doc, score in xr.items() if score > threshold}
    #xr_filtered_sorted = dict(sorted(xr_filtered.items(), key=lambda x: x[1], reverse=True))

    #print("***************************************")
    #print("***************************************")
    #return list(xr.keys())
    return list(xr.items())

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer, util
import time
from tqdm import tqdm
for i in list_of_models :
    model = CrossEncoder(i, max_length=256)
    run = load_run("mrTydi_bm25_6faiss_rrf.txt", 1000)
    output_run= i.split("/")[1]+"_pseudo_judgment.tsv"
    print(output_run)
    #start = time.time()
    for k in tqdm(run.keys()):
        run[k] = reranking_crossEncoder(k, run[k], 0.9)
        #print(type(run[k]))    
        #break
        #end = time.time()
        #print(end - start)
    # Run reranker
    with open(output_run, 'w', encoding='utf-8') as f_out:
        for k in run.keys():
            rank=1
            for j in run[k]:
                f_out.write(f'{k}\t{j[0]}\t{rank}\t{j[1]}\n')
                rank =rank +1
                
    print("Done!")